In [1]:
from PyXRAIN import composite
from datetime import datetime, timedelta
from netCDF4 import Dataset
import numpy as np
import xarray as xr
import pandas as pd

In [2]:
# Open input files
start_time = datetime(2025, 11, 1, 0, 0)
area_name = "TOHOKU0001"
input_prefix  = "/mnt/f/DATA/"
#output_prefix = "/mnt/f/DATA/XRAIN/nc/"
output_prefix = "./"

In [3]:
pres_time = start_time
pres_time = pres_time + timedelta(minutes=1)
data_path=datetime.strftime(pres_time, "%Y/%m/%d/%H")
file_name_part=datetime.strftime(pres_time, "%Y%m%d-%H%M")

input_file = input_prefix+"XRAIN/composite_cx/"+area_name+"/"+data_path+"/"+area_name+"-"+file_name_part+"-G000-EL000000"
print("Input file:", input_file)

radar = composite(input_file, False)
x,y = radar.center
rain = radar.comp
edge = radar.edge
print("Data range")
print("N"+str(np.round(radar.edge[0][0],decimals=3))+"-"+str(np.round(radar.edge[0][1],decimals=3))+"deg ("+str(rain.shape[1])+")")
print("E"+str(np.round(radar.edge[1][0],decimals=3))+"-"+str(np.round(radar.edge[1][1],decimals=3))+"deg ("+str(rain.shape[0])+")")


Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0001-G000-EL000000
Data range
N135.0-148.0deg (4160)
E36.0-42.0deg (2880)


In [4]:
# create CF-compliant netCDF with time in "days since 2000-01-01 00:00:00"

file_name_date=datetime.strftime(pres_time, "%Y%m%d")
output_ncfile = output_prefix+"XRAIN_"+area_name+"_"+file_name_date+".nc"
target_time = pd.to_datetime(pres_time)
epoch = pd.Timestamp("2000-01-01T00:00:00")
time_days = (target_time - epoch) / pd.Timedelta(days=1)
lat = y[:, 0]
lon = x[0, :]

# convert to float32 and replace -1 with NaN (fill value -> missing)
rain = rain.astype(np.float32)
rain[rain == -1] = np.nan

ds = xr.Dataset(
    data_vars={
        "precipitation": (["time", "lat", "lon"], rain[np.newaxis, ...], {
            "units": "mm/h",
            "long_name": "precipitation Rate"
        })
    },
    coords={
        "time": (["time"], np.array([time_days], dtype=float)),
        "lat": (["lat"], lat, {"units": "degrees_north"}),
        "lon": (["lon"], lon, {"units": "degrees_east"})
    },
    attrs={"Conventions": "CF-1.8"}
)

ds.time.attrs["units"] = "days since 2000-01-01 00:00:00"
ds.time.attrs["calendar"] = "gregorian"

ds.to_netcdf(output_ncfile, mode="w", unlimited_dims=["time"])
print(f"Saved {output_ncfile} (time={time_days} days since 2000-01-01)")
ds.close()

Saved ./XRAIN_TOHOKU0001_20251101.nc (time=9436.000694444445 days since 2000-01-01)


In [5]:
pres_time = start_time

for i in range(24*60-1):
    pres_time = pres_time + timedelta(minutes=1)
    data_path=datetime.strftime(pres_time, "%Y/%m/%d/%H")
    file_name_part=datetime.strftime(pres_time, "%Y%m%d-%H%M")

    target_time = pd.to_datetime(pres_time)
    time_days = (target_time - epoch) / pd.Timedelta(days=1)

    
    input_file = input_prefix+"XRAIN/composite_cx/"+area_name+"/"+data_path+"/"+area_name+"-"+file_name_part+"-G000-EL000000"
    print("Input file:", input_file)
    radar = composite(input_file, False)
    rain = radar.comp
    
    # convert to float32 and replace -1 with NaN (fill value -> missing)
    rain = rain.astype(np.float32)
    rain[rain == -1] = np.nan
    try:
        with Dataset(output_ncfile, mode="a") as nc:
            current_size = len(nc.dimensions["time"])
            nc.variables["time"][current_size] = time_days
            nc.variables["precipitation"][current_size, :, :] = rain[np.newaxis, ...]
    except Exception as e:
        print(f"Error appending data for time {time_days} days: {e}")
    

Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0001-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0002-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0003-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0004-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0005-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0006-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0007-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0008-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composite_cx/TOHOKU0001/2025/11/01/00/TOHOKU0001-20251101-0009-G000-EL000000
Input file: /mnt/f/DATA/XRAIN/composi

KeyboardInterrupt: 